## El siguiente es un instructivo paso a paso para crear una API en local.

Vale decir que es un proceso similar para un servidor, podriamos tranquilamente publicar desde nuestro pc el puerto y tendriamos un servidor corriendo nuestras API sin costo alguno.

##### EL SIGUIENTE CODIGO NO ESTA DISEÑADO PARA EJECUTARSE EN COLAB, SON PASOS PARA EJECUTARSE EN EL ENTORNO LOCAL

Si bien no es obligatorio, es altamente recomendable que siempre que vamos a trabajar sobre un proyecto nuevo, que requiera la instalación de librerias, lo encapsulemos dentro de un entorno virtual, para esto comenzamos creando nuestro directorio de trabajo para el laboratorio y ejecutamos

> python -m venv flask

python -m  sirve para llamar un comando de python, venv es el comando que nos permite crear entornos virtuales,  y flask es el nombre que le puse a mi entorno

<br>

>source flask/bin/activate

Activamos el entorno que acabamos de crear, vamos a ver que el promt de nuestra consola cambiara y en mi caso me va a poner (flask).

<br>

>pip install flask

En el supuesto que no tengan instalada la libreria flask la instalamos con el comando anterior.

>gedit app.py

Vamos a crear un archivo con el codigo que se va a ejecutar para tener nuestra api funcionando.  Se puede generar el archivo con cualquier editor, sea VSCODE , notepad++, o algo tan simple como vi o gedit.

<br>

En la siguiente celda paso todo el codigo para pegar en el archivo creado


In [ ]:

# ------------------------------------------------------------------------------------------#
#   El siguiente codigo es parte de un laboratorio para explicar como funciona una API
#   En este caso el cliente podra hacer una llamada con una oración
#   El servidor va a responder con 2 datos
#   1) Cantidad de palabras en la oración
#   2) La misma oracion pero con todos los caracteres en mayuscula
# ------------------------------------------------------------------------------------------#


# Importamos librerias
from flask import Flask, request, jsonify

# jsonify es una libreria que nos permite trabajar con el formato json y se encarga de gestionar las cabeceras de respuesta para que se identifique correctamente como json

# Inicializamos la aplicación Flask
app = Flask(__name__)

# Definimos la ruta de la API (endpoint) y el método (POST es ideal para enviar datos)
@app.route('/procesar', methods=['POST'])
def procesar_oracion():
    # Recibimos los datos en formato JSON
    datos = request.get_json()

    # Validamos que nos hayan enviado la oración
    if not datos or 'oracion' not in datos:
        return jsonify({'error': 'Por favor, proporciona una oración bajo la clave "oracion".'}), 400

    oracion_original = datos['oracion']

    # Procesamos la información
    en_mayusculas = oracion_original.upper()
    cantidad_palabras = len(oracion_original.split())

    # Armamos la respuesta en JSON
    respuesta = {
        'oracion_original': oracion_original,
        'oracion_mayuscula': en_mayusculas,
        'cantidad_palabras': cantidad_palabras
    }

    return jsonify(respuesta)

# Ejecutamos el servidor local
if __name__ == '__main__':
    app.run(debug=True)

<br>


>python app.py

Ejecutamos nuestro servidor flask disponibilizando un puerto y un endpoint

Vamos a obtener una respuesta en consola similar a esta:

<br>

```
 * Serving Flask app 'app'
 * Debug mode: on
WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat
 * Debugger is active!
 * Debugger PIN: 202-958-181
127.0.0.1 - - [23/Jun/2026 10:27:38] "POST /procesar HTTP/1.1" 200 -
127.0.0.1 - - [23/Jun/2026 10:28:20] "POST /procesar HTTP/1.1" 200 -
```

<br>

Esto significa que en nuestro localhost ya tenemos habilitado el puerto 5000 para escuchar solicitudes, y particularmente tenemos un endpoint habilitado esperando un llamado.

Cuando hicimos:
<br>

> @app.route('/procesar', methods=['POST'])


<br>

Definimos que el endpoint /procesar estara escuchando solicitudes con un formato especifico.

Ahora podemos probarlo desde otra consola


<br>

```
 curl -X POST http://127.0.0.1:5000/procesar -H "Content-Type: application/json" -d "{\"oracion\": \"Correr de noche te permite mantenerte concentrado por mas tiempo\"}"
{
  "cantidad_palabras": 10,
  "oracion_mayuscula": "CORRER DE NOCHE TE PERMITE MANTENERTE CONCENTRADO POR MAS TIEMPO",
  "oracion_original": "Correr de noche te permite mantenerte concentrado por mas tiempo"
}
```


<br>

La función curl nos permite hacer un llamado POST a **http://127.0.0.1:5000/procesar**, que como dijimos es nuestro localhost:puerto 5000 / endpoint procesar

<br>

Debimos aclararle la cabecera -H "Conten-Type: application/json" para que supuera como debe interpretar el contenido que le estamos pasando, y luego le pasamos la oración que nos interesa analizar.

Acción siguiente obtenemos la respuesta en formato JSON con tres elementos:

1) cantidad_palabras
2) oracion_mayuscula
3) oracion_original


Con esto podemos decir que ya tenemos la estructura minima de una API Rest funcionando en local.

Ahora vamos a llevarlo a un escenario mas tipico, donde un cliente realiza una consulta al servidor, entonces vamos a crear un archivo

> cliente.py

Este archivo al ser ejecutado nos va a pedir que introduzcamos un texto y se encargara de la comunicación con la api que hicimos antes para devolvernos el texto reformateado

In [ ]:
import requests

def consultar_api():
    # URL donde está corriendo tu API de Flask
    url = "http://127.0.0.1:5000/procesar"

    print("--- Cliente de la API de Procesamiento ---")
    # Pedimos el texto al usuario por consola
    oracion_usuario = input("Ingresa una oración para analizar: ")

    # Validamos que no esté vacía
    if not oracion_usuario.strip():
        print("No ingresaste ningún texto.")
        return

    # Armamos el paquete de datos (el diccionario que se convertirá en JSON)
    datos = {"oracion": oracion_usuario}

    print("\nEnviando datos a la API...")

    try:
        # Hacemos la petición POST enviando los datos como JSON
        respuesta = requests.post(url, json=datos)

        # Si la API responde con un código de éxito (200)
        if respuesta.status_code == 200:
            # Convertimos la respuesta JSON en un diccionario de Python
            resultado = respuesta.json()

            # Mostramos los datos bien formateados y limpios
            print("\n========================================")
            print("         RESULTADO DE LA API            ")
            print("========================================")
            print(f"Texto original:   {resultado['oracion_original']}")
            print(f"Texto en MAYUS:   {resultado['oracion_mayuscula']}")
            print(f"Total palabras:   {resultado['cantidad_palabras']}")
            print("========================================")
        else:
            print(f"\nError en la API. Código de estado: {respuesta.status_code}")
            print(respuesta.json())

    except requests.exceptions.ConnectionError:
        print("\n[ERROR] No se pudo conectar con la API.")
        print("¿Asegúrate de que 'app.py' se está ejecutando en otra terminal?")

if __name__ == "__main__":
    consultar_api()

Lo que vamos a realizar es poner a correr el servidor con **app.py**, en mi caso particular lo deje corriendo en un segundo plano y ejecute el script **cliente.py**



```
(flask) (base) ariel@meragelman:~/Escritorio/CURSOS/CURSOS/01-IA/04-API/PRACTICAS/laboratorio-api-1$ python app.py
 * Serving Flask app 'app'
 * Debug mode: on
WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat
 * Debugger is active!
 * Debugger PIN: 202-958-181

^Z
[1]+  Detenido                python app.py
(flask) (base) ariel@meragelman:~/Escritorio/CURSOS/CURSOS/01-IA/04-API/PRACTICAS/laboratorio-api-1$ bg 1
[1]+ python app.py &
(flask) (base) ariel@meragelman:~/Escritorio/CURSOS/CURSOS/01-IA/04-API/PRACTICAS/laboratorio-api-1$ python cliente.py
--- Cliente de la API de Procesamiento ---
Ingresa una oración para analizar: esta es una oracion simple de prueba

Enviando datos a la API...
127.0.0.1 - - [23/Jun/2026 17:53:19] "POST /procesar HTTP/1.1" 200 -

========================================
         RESULTADO DE LA API            
========================================
Texto original:   esta es una oracion simple de prueba
Texto en MAYUS:   ESTA ES UNA ORACION SIMPLE DE PRUEBA
Total palabras:   7
========================================

```

